In [1]:
# ============================================================================
# MINIMAL VERSION: Rank Collapse Analysis with Dataset Size Comparison
# Ready to run - will generate the requested plot automatically
# ============================================================================

import os
import pickle
import time
from pathlib import Path
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict
from itertools import product
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

print("✓ Imports successful!")

# Setup directories
BASE_DIR = Path('/kaggle/working') if os.path.exists('/kaggle/working') else Path('.')
PLOTS_DIR = BASE_DIR / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"✓ Device: {DEVICE}")
print(f"✓ Plots directory: {PLOTS_DIR}")

# ============================================================================
# CIFAR-10 Global Cache
# ============================================================================

_CIFAR_CACHE = {'train': None, 'test': None}

def _initialize_cifar_cache(root: str = './data', device: str = 'cuda'):
    global _CIFAR_CACHE
    if _CIFAR_CACHE['train'] is not None:
        return
    
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2023, 0.1994, 0.2010)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    
    try:
        train_full = torchvision.datasets.CIFAR10(root=root, train=True, download=True, transform=transform)
        test_full = torchvision.datasets.CIFAR10(root=root, train=False, download=True, transform=transform)
    except:
        train_full = torchvision.datasets.CIFAR10(root=root, train=True, download=False, transform=transform)
        test_full = torchvision.datasets.CIFAR10(root=root, train=False, download=False, transform=transform)
    
    train_data = []
    train_labels = []
    for img, label in train_full:
        train_data.append(img.view(-1))
        train_labels.append(label)
    
    test_data = []
    test_labels = []
    for img, label in test_full:
        test_data.append(img.view(-1))
        test_labels.append(label)
    
    _CIFAR_CACHE['train'] = {
        'X': torch.stack(train_data).to(device),
        'y': torch.tensor(train_labels).to(device)
    }
    _CIFAR_CACHE['test'] = {
        'X': torch.stack(test_data).to(device),
        'y': torch.tensor(test_labels).to(device)
    }
    
    print(f"✓ CIFAR-10 cached: Train={len(train_labels)}, Test={len(test_labels)}")

print("\n" + "="*80)
print("📦 INITIALIZING CIFAR-10 CACHE")
print("="*80)
_initialize_cifar_cache(device=DEVICE)
print("="*80 + "\n")

class CIFAR10Subset(Dataset):
    def __init__(self, n_per_class: int, train: bool = True, seed: int = 42):
        global _CIFAR_CACHE
        cache_key = 'train' if train else 'test'
        X_full = _CIFAR_CACHE[cache_key]['X']
        y_full = _CIFAR_CACHE[cache_key]['y']
        
        if train and n_per_class > 0:
            selected_indices = []
            rng = np.random.RandomState(seed)
            for k in range(10):
                class_indices = torch.where(y_full == k)[0].cpu().numpy()
                rng.shuffle(class_indices)
                selected_indices.extend(class_indices[:min(n_per_class, len(class_indices))])
            self.X = X_full[selected_indices]
            self.y = y_full[selected_indices]
        else:
            self.X = X_full
            self.y = y_full

    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ============================================================================
# Network Architecture
# ============================================================================

class BottleneckMLP(nn.Module):
    def __init__(self, d0: int = 1000, K: int = 10, L_hidden: int = 6, d_b: int = 64):
        super().__init__()
        self.d0 = d0
        self.K = K
        self.L_hidden = L_hidden
        self.d_b = d_b
        
        self.projector = nn.Sequential(
            nn.Linear(3072, d0),
            nn.ReLU()
        )
        
        layers = []
        widths = [d0] + [d_b] * L_hidden + [K]
        for i in range(len(widths) - 1):
            layers.append(nn.Linear(widths[i], widths[i+1]))
            if i < len(widths) - 2:
                layers.append(nn.ReLU())
        
        self.network = nn.Sequential(*layers)
        self.widths = widths
    
    def forward(self, x):
        h = self.projector(x)
        return self.network(h)
    
    def get_layer_activations(self, x: torch.Tensor) -> List[torch.Tensor]:
        h = self.projector(x)
        activations = [h.detach()]
        
        for layer in self.network:
            h = layer(h)
            if isinstance(layer, nn.Linear):
                activations.append(h.detach())
        
        return activations

# ============================================================================
# Metrics - FIXED VERSION
# ============================================================================

def stable_rank(H: torch.Tensor) -> float:
    """
    Stable rank: ||H||_F^2 / ||H||_2^2
    
    CRITICAL FIXES:
    - Dead signal (spec_norm=0) returns 1.0, NOT width
    - Tiny batches return 1.0, NOT width
    - NaN/Inf returns np.nan, NOT width
    - Never returns width as fallback
    """
    try:
        # FIX 1: Tiny batch size (physically rank 1)
        if H.shape[0] < 2:
            return 1.0  # Not width!
        
        # Compute norms
        frob_norm = torch.norm(H, p='fro').item() ** 2
        spec_norm = torch.linalg.matrix_norm(H, ord=2).item() ** 2
        
        # FIX 2: Dead Signal Bug (THE BIG ONE - was returning width)
        # If spectral norm is 0, matrix is dead → rank is 1
        if spec_norm < 1e-10:
            return 1.0  # Collapsed, not full rank!
        
        # Check for NaN/Inf in intermediate values
        if not np.isfinite(frob_norm) or not np.isfinite(spec_norm):
            return np.nan  # Don't pretend it's valid
        
        # Compute stable rank
        rank = frob_norm / (spec_norm + 1e-10)
        
        # FIX 3: NaN handling in final result
        if not np.isfinite(rank):
            return np.nan  # Error state, not collapse
        
        # Clamp to valid range [1.0, min(H.shape)]
        rank = max(1.0, min(rank, float(min(H.shape))))
        
        return rank

    except Exception as e:
        print(f"⚠️  stable_rank failed: {e}")
        return np.nan  # Error → NaN, not width


def compute_metrics(model: nn.Module, X_train: torch.Tensor, y_train: torch.Tensor, K: int) -> Dict:
    """Compute metrics with proper error handling."""
    model.eval()
    with torch.no_grad():
        activations = model.get_layer_activations(X_train)
        metrics = {}
        L = len(activations) - 1
        
        for ell in range(1, L + 1):
            H = activations[ell]
            if len(H.shape) == 1:
                H = H.unsqueeze(0)
            if ell == L or H.shape[1] == K:
                continue
            
            metrics[f'layer_{ell}'] = {
                'stable_rank': stable_rank(H),
                'width': H.shape[1]
            }
    
    return metrics

# ============================================================================
# Configuration
# ============================================================================

@dataclass
class ExperimentConfig:
    K: int = 10
    d0: int = 1000
    n_per_class: int = 50
    d_b: int = 64
    L_hidden: int = 6
    wd: float = 1e-3
    lr: float = 0.05
    max_iters: int = 50000  # Short for testing
    seed: int = 42

# ============================================================================
# Training
# ============================================================================

def train_single_run(config: ExperimentConfig) -> Optional[Dict]:
    torch.manual_seed(config.seed)
    device = DEVICE
    
    try:
        train_full = CIFAR10Subset(config.n_per_class, train=True, seed=config.seed)
        N = len(train_full)
        n_val = max(2, min(10, int(0.1 * N)))
        n_train = N - n_val
        
        train_ds, val_ds = random_split(
            train_full, [n_train, n_val],
            generator=torch.Generator().manual_seed(config.seed)
        )
        
        X_train = torch.stack([train_ds[i][0] for i in range(len(train_ds))])
        y_train = torch.stack([train_ds[i][1] for i in range(len(train_ds))])
        X_val = torch.stack([val_ds[i][0] for i in range(len(val_ds))])
        y_val = torch.stack([val_ds[i][1] for i in range(len(val_ds))])
        
        model = BottleneckMLP(d0=config.d0, K=config.K, L_hidden=config.L_hidden, d_b=config.d_b).to(device)
        optimizer = torch.optim.SGD(model.parameters(), lr=config.lr, momentum=0.9, weight_decay=config.wd)
        criterion = nn.CrossEntropyLoss()
        
        model.train()
        for step in range(config.max_iters):
            optimizer.zero_grad(set_to_none=True)
            logits = model(X_train)
            loss = criterion(logits, y_train)
            loss.backward()
            optimizer.step()
        
        metrics = compute_metrics(model, X_train, y_train, config.K)
        
        result = {
            'config': asdict(config),
            'metrics': metrics
        }
        
        del model, optimizer
        return result
        
    except Exception as e:
        print(f"❌ Run failed: {e}")
        return None

# ============================================================================
# NEW PLOT: Rank Collapse by Dataset Size
# ============================================================================

def plot_rank_collapse_by_dataset_size(results: List[Dict], save_path: Path):
    """
    Shows rank collapse across layers for different dataset sizes.
    X-axis: Layer number
    Y-axis: Stable rank
    Multiple lines: Different n_per_class values
    """
    if len(results) == 0:
        print("⚠️ No results for rank collapse plot")
        return
    
    # Group by n_per_class
    n_groups = {}
    for res in results:
        cfg = res['config']
        n = cfg['n_per_class']
        if n not in n_groups:
            n_groups[n] = []
        n_groups[n].append(res)
    
    if len(n_groups) == 0:
        print("⚠️ No valid groups for rank collapse plot")
        return
    
    # Create figure
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    
    # Sort by dataset size
    sorted_n = sorted(n_groups.keys())
    
    # Color palette
    colors = plt.cm.viridis(np.linspace(0, 1, len(sorted_n)))
    
    for idx, n in enumerate(sorted_n):
        group = n_groups[n]
        
        # Aggregate ranks across all runs for this n
        layer_ranks = {}
        for res in group:
            metrics = res.get('metrics', {})
            for layer_name, layer_metrics in metrics.items():
                if layer_name.startswith('layer_'):
                    layer_idx = int(layer_name.split('_')[1])
                    rank = layer_metrics.get('stable_rank', np.nan)
                    if np.isfinite(rank):
                        if layer_idx not in layer_ranks:
                            layer_ranks[layer_idx] = []
                        layer_ranks[layer_idx].append(rank)
        
        if not layer_ranks:
            continue
        
        # Compute mean and std
        layers = sorted(layer_ranks.keys())
        means = [np.mean(layer_ranks[l]) for l in layers]
        stds = [np.std(layer_ranks[l]) for l in layers]
        
        # Plot line with confidence interval
        ax.plot(layers, means, 'o-', linewidth=2.5, color=colors[idx], 
                label=f'n={n} ({n*10} total samples)', markersize=6, alpha=0.8)
        ax.fill_between(layers, 
                        np.array(means) - np.array(stds), 
                        np.array(means) + np.array(stds), 
                        alpha=0.15, color=colors[idx])
    
    ax.set_xlabel('Layer Number', fontsize=14, fontweight='bold')
    ax.set_ylabel('Stable Rank', fontsize=14, fontweight='bold')
    ax.set_title('Rank Collapse Across Network Depth for Different Dataset Sizes', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.legend(loc='best', fontsize=11, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Add annotation
    ax.text(0.02, 0.98, 'Lower rank = More compression\nBottleneck severity increases with depth', 
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("\n" + "="*80)
print("🚀 RUNNING RANK COLLAPSE EXPERIMENT")
print("="*80)

# Define sweep: Test different dataset sizes
sweep_params = {
    'n_per_class': [5, 10, 20, 50, 100, 500, 5000],  # Different dataset sizes
    'd_b': [64],                           # Fixed bottleneck width
    'L_hidden': [6],                       # Fixed depth
    'wd': [1e-3],
    'seed': [42, 43, 44]                       # 2 seeds for stability
}

# Run experiments
all_results = []
total_runs = np.prod([len(v) for v in sweep_params.values()])
run_count = 0

for combo in product(*sweep_params.values()):
    config_dict = dict(zip(sweep_params.keys(), combo))
    config = ExperimentConfig(**config_dict)
    
    run_count += 1
    print(f"\n[{run_count}/{total_runs}] n={config.n_per_class}, seed={config.seed}")
    
    result = train_single_run(config)
    if result is not None:
        all_results.append(result)
        print(f"  ✓ Completed")

print(f"\n{'='*80}")
print(f"✓ Finished {len(all_results)} runs")
print(f"{'='*80}\n")

# Generate the plot
if len(all_results) > 0:
    print("📊 Generating rank collapse plot...")
    plot_rank_collapse_by_dataset_size(all_results, PLOTS_DIR / 'rank_collapse_by_dataset_size.png')
    
    # Save results
    with open(CHECKPOINT_DIR / 'results.pkl', 'wb') as f:
        pickle.dump(all_results, f)
    print(f"💾 Results saved to {CHECKPOINT_DIR / 'results.pkl'}")
    
    print("\n" + "="*80)
    print("✅ EXPERIMENT COMPLETE!")
    print(f"📊 Plot saved to: {PLOTS_DIR / 'rank_collapse_by_dataset_size.png'}")
    print("="*80)
else:
    print("❌ No results to plot")

✓ Imports successful!
✓ Device: cuda
✓ Plots directory: /kaggle/working/plots

📦 INITIALIZING CIFAR-10 CACHE


100%|██████████| 170M/170M [00:18<00:00, 9.00MB/s]


✓ CIFAR-10 cached: Train=50000, Test=10000


🚀 RUNNING RANK COLLAPSE EXPERIMENT

[1/21] n=5, seed=42
  ✓ Completed

[2/21] n=5, seed=43
  ✓ Completed

[3/21] n=5, seed=44
  ✓ Completed

[4/21] n=10, seed=42
  ✓ Completed

[5/21] n=10, seed=43
  ✓ Completed

[6/21] n=10, seed=44
  ✓ Completed

[7/21] n=20, seed=42
  ✓ Completed

[8/21] n=20, seed=43
  ✓ Completed

[9/21] n=20, seed=44
  ✓ Completed

[10/21] n=50, seed=42
  ✓ Completed

[11/21] n=50, seed=43
  ✓ Completed

[12/21] n=50, seed=44
  ✓ Completed

[13/21] n=100, seed=42
  ✓ Completed

[14/21] n=100, seed=43
  ✓ Completed

[15/21] n=100, seed=44
  ✓ Completed

[16/21] n=500, seed=42
  ✓ Completed

[17/21] n=500, seed=43
  ✓ Completed

[18/21] n=500, seed=44
  ✓ Completed

[19/21] n=5000, seed=42
  ✓ Completed

[20/21] n=5000, seed=43
  ✓ Completed

[21/21] n=5000, seed=44
  ✓ Completed

✓ Finished 21 runs

📊 Generating rank collapse plot...
✓ Saved: rank_collapse_by_dataset_size.png
💾 Results saved to /kaggle/working/checkpoin